# EV Charging Behavior Analysis & Energy Prediction

## 1. Business Problem Statement

Electric Vehicle (EV) adoption is growing rapidly, and charging infrastructure providers need to understand
charging behavior to optimize station usage, manage energy load, and improve user experience.

This project analyzes EV charging session data to understand key patterns in charging behavior — such as
energy consumption, charging duration, and station usage — and builds a machine learning model to predict
energy consumption for a given charging session based on session characteristics (vehicle type, charger type,
time of day, battery capacity, etc.).

## 2. Goal

- Understand what factors most influence EV charging energy consumption and duration.
- Build and compare multiple machine learning models to predict energy consumption per session.
- Track and compare model experiments using MLflow.
- Demonstrate an end-to-end ML pipeline: data ingestion, cleaning, feature engineering, model training,
  evaluation, and automation — similar to production ML pipelines used in real-world industrial settings.

## 3. Approach

1. **Data Ingestion** — Load raw EV charging session data into Databricks.
2. **Exploratory Data Analysis (EDA)** — Understand data distributions, relationships, and quality issues.
3. **Data Cleaning & Feature Engineering** — Prepare data for modeling.
4. **Model Training** — Train and compare multiple regression models to predict Energy Consumed (kWh).
5. **Model Evaluation** — Compare models using standard regression metrics (RMSE, MAE, R²).
6. **Experiment Tracking** — Use MLflow to log and compare all model runs.
7. **Automation** — Package the pipeline into a Databricks Job for scheduled execution.

In [0]:
# Extract the data
df = spark.table("dbacademy.default.ev_charging_patterns")
df.show(10)

## Phase 3: Exploratory Data Analysis (EDA)

Before cleaning or modeling the data, we first explore it to understand its structure, quality, and
underlying patterns. This phase answers key questions:

- What are the data types of each column, and are they correctly inferred?
- How many rows and columns does the dataset contain?
- Are there missing or null values, and in which columns?
- What do the numeric columns look like statistically (min, max, mean, distribution)?
- What are the most common values in key categorical columns (Vehicle Model, Charger Type, User Type)?
- Are there any relationships between features that stand out (e.g., does Battery Capacity relate to
  Energy Consumed)?

**Steps in this phase:**
1. Schema check — confirm data types
2. Row and column count
3. Missing/null value check
4. Summary statistics for numeric columns
5. Value counts for key categorical columns
6. Basic visualizations of important trends
7. Initial observations summary

In [0]:
# step 1:- Schema Check
df.printSchema()
# This function is used to shows your structure of your dataframe specifically every column's name and datatype

In [0]:
# step 2:- Row and columns count 
row_count = df.count()
col_count = len(df.columns)
print(f"Number of rows: {row_count}")
print(f"Number of columns: {col_count}")
print(f"Columns: {df.columns}")

In [0]:
# Step 3:- Missing/Null Value check
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
])

null_counts.show(vertical=True)

In [0]:
# Check if the same rows are missing across all three columns
df.filter(
    col("Energy Consumed (kWh)").isNull() |
    col("Charging Rate (kW)").isNull() |
    col("Distance Driven (since last charge) (km)").isNull()
).select(
    "User ID", "Energy Consumed (kWh)", "Charging Rate (kW)", "Distance Driven (since last charge) (km)"
).show(20)

In [0]:
total_rows = df.count()
print(f"Total rows: {total_rows}")
print(f"Missing Energy Consumed: 66 ({66/total_rows*100:.2f}%)")
print(f"Missing Charging Rate: 66 ({66/total_rows*100:.2f}%)")
print(f"Missing Distance Driven: 66 ({66/total_rows*100:.2f}%)")

In [0]:
# Step 4:- Sumamry statistics for numeric columns
numeric_cols = [
    "Battery Capacity (kWh)",
    "Energy Consumed (kWh)",
    "Charging Duration (hours)",
    "Charging Rate (kW)",
    "Charging Cost (USD)",
    "State of Charge (Start %)",
    "State of Charge (End %)",
    "Distance Driven (since last charge) (km)",
    "Temperature (°C)",
    "Vehicle Age (years)"
]
df.select(numeric_cols).describe().show()
# Calculates count, mean, standard deviation (stddev), min, and max for each selected numeric column

In [0]:
# step 5:- Value counts for categorical coulmns 
print("Vehicle Model Distribution:")
df.groupBy("Vehicle Model").count().orderBy("count", ascending=False).show()

print("Charger Type distribution:")
df.groupBy("Charger Type").count().orderBy("count", ascending=False).show()

print("User Type distribution:")
df.groupBy("User Type").count().orderBy("count", ascending=False).show()

print("Time of Day distribution:")
df.groupBy("Time of Day").count().orderBy("count", ascending=False).show()

print("Day of Week distribution:")
df.groupBy("Day of Week").count().orderBy("count", ascending=False).show()

In [0]:
# Step 6:- Correlation between Numeric Features
from pyspark.sql.functions import corr
correlation_pairs = [
    ("Battery Capacity (kWh)", "Energy Consumed (kWh)"),
    ("Charging Duration (hours)", "Energy Consumed (kWh)"),
    ("Charging Rate (kW)", "Energy Consumed (kWh)"),
    ("State of Charge (Start %)", "Energy Consumed (kWh)"),
    ("Temperature (°C)", "Energy Consumed (kWh)"),
    ("Vehicle Age (years)", "Energy Consumed (kWh)"),
    ("Distance Driven (since last charge) (km)", "Energy Consumed (kWh)") 
]
for col1, col2 in correlation_pairs:
    correlation_value = df.select(corr(col1, col2)).collect()[0][0]
    print(f"Correlation between {col1} and {col2}: {correlation_value:.4f}")

In [0]:
# step 7:- Average energu consumed by charger type 
display(
    df.groupBy("Charger Type")
    .avg("Energy Consumed (kWh)")
    .orderBy("Charger Type")
)

In [0]:
# EDA - Step 7b: Sessions by Time of Day
display(
    df.groupBy("Time of Day")
      .count()
      .orderBy("Time of Day")
)

In [0]:
# EDA - Step 7c: Energy Consumed vs Charging Duration
display(
    df.select("Charging Duration (hours)", "Energy Consumed (kWh)")
)

## Phase 3 Summary: EDA Findings

**Dataset overview:**
- Total rows: [fill in from Step 3.2]
- Total columns: 19

**Data quality:**
- Three columns contain missing values, each around 66 rows (~[X]% of total data):
  - Energy Consumed (kWh)
  - Charging Rate (kW)
  - Distance Driven (since last charge) (km)
- Missing values are scattered independently across different rows, not clustered together.
- No missing values in any other column.

**Numeric feature summary:**
- [Note any interesting min/max findings — e.g., "State of Charge values stay within expected 0-100% range" or flag if anything looked off]

**Categorical distributions:**
- Most common Vehicle Model: [fill in]
- Most common Charger Type: [fill in]
- Most common User Type: [fill in]

**Correlation findings:**
- [Fill in which features showed the strongest correlation with Energy Consumed — this will guide feature selection for modeling]

**Next steps:**
- Handle missing values in the three affected columns (Phase 4: Cleaning)
- Engineer new features from Charging Start Time (hour, day of week patterns)
- Encode categorical variables for modeling
- Proceed to feature engineering and model training


## Phase 4: Data Cleaning

Based on our EDA findings, this phase addresses the data quality issues we identified before moving into
feature engineering and modeling.

**Issues to address:**
1. Missing values in three numeric columns (Energy Consumed, Charging Rate, Distance Driven) — approximately
   66 rows each, scattered independently.
2. Check for duplicate rows.
3. Validate value ranges (e.g., State of Charge should be between 0-100%, durations should be positive).
4. Confirm data types are correct for downstream processing.

**Approach:**
- Fill missing numeric values using the median of each column (robust to outliers, appropriate for
  sensor-like continuous data).
- Remove exact duplicate rows, if any exist.
- Flag and review any out-of-range values.

In [0]:
# CLEANING - Step 1: Fill missing values with median
from pyspark.sql.functions import expr

columns_to_fill = [
    "Energy Consumed (kWh)",
    "Charging Rate (kW)",
    "Distance Driven (since last charge) (km)"
]

for column_name in columns_to_fill:
    median_value = df.approxQuantile(column_name, [0.5], 0.01)[0]
    df = df.fillna({column_name: median_value})
    print(f"Filled '{column_name}' nulls with median: {median_value:.2f}")

In [0]:
# Verify no more nulls in these columns
from pyspark.sql.functions import col, sum as spark_sum
null_check = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in columns_to_fill
])
null_check.show()

In [0]:
# CLEANING - Step 2: Check for duplicate rows
total_rows = df.count()
distinct_rows = df.distinct().count()
duplicate_count = total_rows - distinct_rows

print(f"Total rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Duplicate rows: {duplicate_count}")

In [0]:
# CLEANING - Step 3: Validate value ranges

# Check State of Charge stays within 0-100%
invalid_soc_start = df.filter(
    (col("State of Charge (Start %)") < 0) | (col("State of Charge (Start %)") > 100)
).count()

invalid_soc_end = df.filter(
    (col("State of Charge (End %)") < 0) | (col("State of Charge (End %)") > 100)
).count()

# Check for negative durations, energy, or cost (shouldn't be negative)
invalid_duration = df.filter(col("Charging Duration (hours)") < 0).count()
invalid_energy = df.filter(col("Energy Consumed (kWh)") < 0).count()
invalid_cost = df.filter(col("Charging Cost (USD)") < 0).count()

print(f"Invalid State of Charge (Start %): {invalid_soc_start}")
print(f"Invalid State of Charge (End %): {invalid_soc_end}")
print(f"Negative Charging Duration: {invalid_duration}")
print(f"Negative Energy Consumed: {invalid_energy}")
print(f"Negative Charging Cost: {invalid_cost}")

In [0]:
# Check actual invalid values for State of Charge
print("Invalid State of Charge (Start %):")
df.filter(
    (col("State of Charge (Start %)") < 0) | (col("State of Charge (Start %)") > 100)
).select("User ID", "State of Charge (Start %)").show()

print("Invalid State of Charge (End %):")
df.filter(
    (col("State of Charge (End %)") < 0) | (col("State of Charge (End %)") > 100)
).select("User ID", "State of Charge (End %)").show()

In [0]:
# CLEANING - Step 3b: Cap State of Charge to valid 0-100% range
from pyspark.sql.functions import when

df = df.withColumn(
    "State of Charge (Start %)",
    when(col("State of Charge (Start %)") > 100, 100)
    .when(col("State of Charge (Start %)") < 0, 0)
    .otherwise(col("State of Charge (Start %)"))
)

df = df.withColumn(
    "State of Charge (End %)",
    when(col("State of Charge (End %)") > 100, 100)
    .when(col("State of Charge (End %)") < 0, 0)
    .otherwise(col("State of Charge (End %)"))
)

print("Values capped successfully.")

In [0]:
# Verify no more invalid values
invalid_soc_start_after = df.filter(
    (col("State of Charge (Start %)") < 0) | (col("State of Charge (Start %)") > 100)
).count()

invalid_soc_end_after = df.filter(
    (col("State of Charge (End %)") < 0) | (col("State of Charge (End %)") > 100)
).count()

print(f"Invalid Start % after fix: {invalid_soc_start_after}")
print(f"Invalid End % after fix: {invalid_soc_end_after}")

## Phase 4 Summary: Data Cleaning

**Actions taken:**
1. Filled missing values in three columns (Energy Consumed, Charging Rate, Distance Driven) using
   median imputation (~66 rows each, scattered independently).
2. Checked for duplicate rows — [fill in result: none found / X duplicates removed].
3. Validated value ranges:
   - Found 9 rows with State of Charge (Start %) above 100% (up to ~152%).
   - Found 23 rows with State of Charge (End %) above 100% (up to ~178%).
   - Capped both columns to a valid 0-100% range rather than dropping rows, to preserve other valid
     data in those rows.
4. Confirmed no negative values in Duration, Energy Consumed, or Cost.

**Result:** Dataset is now clean, complete, and within valid logical ranges, ready for feature
engineering.

## Phase 5: Feature Engineering

This phase transforms our cleaned data into a format suitable for machine learning models. We'll:

1. Extract time-based features from Charging Start Time (hour, is_weekend).
2. Create a derived feature: State of Charge change (End % - Start %).
3. Encode categorical columns (Vehicle Model, Charger Type, User Type, Time of Day, Day of Week) into
   numeric form, since ML models require numeric input.
4. Assemble all features into a single vector column, as required by Spark's MLlib.

In [0]:
# FEATURE ENGINEERING - Step 1: Extract time-based features
from pyspark.sql.functions import hour, dayofweek, when as when_func

df = df.withColumn("Start_Hour", hour(col("Charging Start Time")))
df = df.withColumn("Is_Weekend", when_func(dayofweek(col("Charging Start Time")).isin([1, 7]), 1).otherwise(0))

df.select("Charging Start Time", "Start_Hour", "Is_Weekend").show(10)

In [0]:
# FEATURE ENGINEERING - Step 2: State of Charge change
df = df.withColumn(
    "SoC_Change",
    col("State of Charge (End %)") - col("State of Charge (Start %)")
)

df.select("State of Charge (Start %)", "State of Charge (End %)", "SoC_Change").show(10)

In [0]:
# FEATURE ENGINEERING - Step 3: Encode categorical columns
from pyspark.sql.functions import col as spark_col
from pyspark.ml.feature import StringIndexer

categorical_cols = ["Vehicle Model", "Charger Type", "User Type", "Time of Day", "Day of Week"]

indexers = [
    StringIndexer(inputCol=column, outputCol=column + "_Index")
    for column in categorical_cols
]

for indexer in indexers:
    df = indexer.fit(df).transform(df)

df.select([c + "_Index" for c in categorical_cols]).show(10)

In [0]:
# FEATURE ENGINEERING - Step 4: Assemble features into a vector
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "Battery Capacity (kWh)",
    "Charging Duration (hours)",
    "Charging Rate (kW)",
    "Charging Cost (USD)",
    "State of Charge (Start %)",
    "State of Charge (End %)",
    "SoC_Change",
    "Distance Driven (since last charge) (km)",
    "Temperature (°C)",
    "Vehicle Age (years)",
    "Start_Hour",
    "Is_Weekend",
    "Vehicle Model_Index",
    "Charger Type_Index",
    "User Type_Index",
    "Time of Day_Index",
    "Day of Week_Index"
]

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
df_final = assembler.transform(df)

df_final.select("features").show(5, truncate=False)

## Phase 5 Summary: Feature Engineering

**Features created:**
1. Time-based features from Charging Start Time: Start_Hour, Is_Weekend.
2. Derived feature: SoC_Change (State of Charge End % - Start %).
3. Encoded 5 categorical columns into numeric form using StringIndexer: Vehicle Model, Charger Type,
   User Type, Time of Day, Day of Week.
4. Assembled 17 features into a single vector column ("features") using VectorAssembler, ready for
   Spark ML models.

**Target variable:** Energy Consumed (kWh) — excluded from the feature set, since this is what our
models will predict.

**Result:** df_final now contains a "features" vector column and the target column, ready for
train/test splitting and model training.

## Phase 6: Load

Save the cleaned, feature-engineered dataset as a Delta table, so it can be reliably reused for model
training without repeating the cleaning and feature engineering steps each time.

In [0]:
# LOAD - Save processed data as a Delta table
import re
for old_name in df_final.columns:
    new_name = re.sub(r'[ ,;{}()\n\t=]', '_', old_name)
    if new_name != old_name:
        df_final = df_final.withColumnRenamed(old_name, new_name)

df_final.write.format("delta").mode("overwrite").saveAsTable("dbacademy.default.ev_charging_silver")

print("Saved as Delta table: dbacademy.default.ev_charging_silver")

In [0]:
df_check = spark.table("dbacademy.default.ev_charging_silver")
df_check.printSchema()

In [0]:
# View complete Silver table data
df_silver = spark.table("dbacademy.default.ev_charging_silver")
display(df_silver)

## Phase 7: Model Training

With our cleaned, feature-engineered data saved as a Delta table, we now train machine learning models
to predict Energy Consumed (kWh) for a given charging session.

**Approach:**
1. Split data into training (80%) and test (20%) sets.
2. Train a Linear Regression model as a baseline.
3. Train a Random Forest Regressor as a more complex comparison model.
4. Evaluate both models on the test set in Phase 8.

In [0]:
# MODEL TRAINING - Step 1: Load data and split train/test
df_model = spark.table("dbacademy.default.ev_charging_silver")

train_data, test_data = df_model.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_data.count()}")
print(f"Test rows: {test_data.count()}")

In [0]:
# MODEL TRAINING - Step 2: Train Linear Regression
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="Energy_Consumed__kWh_")

lr_model = lr.fit(train_data)

print("Linear Regression model trained successfully.")
print(f"Coefficients: {lr_model.coefficients}")
print(f"Intercept: {lr_model.intercept}")

In [0]:
# MODEL TRAINING - Step 3: Make predictions with Linear Regression
lr_predictions = lr_model.transform(test_data)

lr_predictions.select("Energy_Consumed__kWh_", "prediction").show(10)

In [0]:
# MODEL TRAINING - Step 4: Train Random Forest Regressor
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(featuresCol="features", labelCol="Energy_Consumed__kWh_", seed=42)

rf_model = rf.fit(train_data)

rf_predictions = rf_model.transform(test_data)
rf_predictions.select("Energy_Consumed__kWh_", "prediction").show(10)

## Phase 8: Model Evaluation

We evaluate both models on the test set using standard regression metrics:

- **RMSE (Root Mean Squared Error)** — average prediction error, in the same units as the target
  (kWh). Lower is better.
- **MAE (Mean Absolute Error)** — average absolute difference between predicted and actual values.
  Lower is better.
- **R² (R-squared)** — proportion of variance in the target explained by the model. Ranges from
  0 to 1 (can be negative for poor models). Higher is better; closer to 1 means the model explains
  the data well.

In [0]:
# EVALUATION - Step 1: Calculate metrics for both models
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(labelCol="Energy_Consumed__kWh_", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="Energy_Consumed__kWh_", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="Energy_Consumed__kWh_", predictionCol="prediction", metricName="r2")

# Linear Regression metrics
lr_rmse = evaluator_rmse.evaluate(lr_predictions)
lr_mae = evaluator_mae.evaluate(lr_predictions)
lr_r2 = evaluator_r2.evaluate(lr_predictions)

# Random Forest metrics
rf_rmse = evaluator_rmse.evaluate(rf_predictions)
rf_mae = evaluator_mae.evaluate(rf_predictions)
rf_r2 = evaluator_r2.evaluate(rf_predictions)

print("Linear Regression:")
print(f"  RMSE: {lr_rmse:.4f}")
print(f"  MAE: {lr_mae:.4f}")
print(f"  R²: {lr_r2:.4f}")

print("\nRandom Forest:")
print(f"  RMSE: {rf_rmse:.4f}")
print(f"  MAE: {rf_mae:.4f}")
print(f"  R²: {rf_r2:.4f}")

## Phase 8 Summary: Model Evaluation

**Results:**

| Model | RMSE | MAE | R² |
|---|---|---|---|
| Linear Regression | 21.74 | 18.68 | 0.0146 |
| Random Forest | 21.88 | 18.66 | 0.0019 |

**Findings:**
- Both models perform poorly, with R² close to 0 — indicating the available features explain almost
  none of the variance in Energy Consumed.
- Linear Regression slightly outperformed Random Forest, suggesting there is little genuine non-linear
  signal for the more complex model to capture.
- This strongly suggests the dataset's Energy Consumed values may be synthetically/randomly generated,
  without a strong underlying relationship to the other features — common in certain practice datasets.

**Conclusion:**
While neither model achieves strong predictive performance, this exercise successfully demonstrates a
complete, honest ML pipeline: data ingestion, cleaning, feature engineering, model training, and
evaluation. In a real-world setting, this result would prompt further investigation into data quality,
additional feature sources, or reconsideration of the prediction target — exactly the kind of pipeline
analysis described in the internship's "analysis of existing pipeline components for performance,
stability and scalability."

## Phase 9: Experiment Tracking with MLflow

We use MLflow to formally track our model experiments — logging parameters, metrics, and the trained
models themselves. This creates a reproducible record of what was tried and how it performed, and
allows easy comparison between runs directly in the MLflow UI.

Even though model performance in this project was weak (see Phase 8), tracking both experiments
properly demonstrates the MLOps workflow used in production ML pipelines.

In [0]:
# MLFLOW - Step 1: Log Linear Regression experiment
import mlflow
import mlflow.spark

with mlflow.start_run(run_name="Linear_Regression_EV_Charging"):
    mlflow.log_param("model_type", "Linear Regression")
    mlflow.log_param("features_used", len(feature_columns))

    mlflow.log_metric("rmse", lr_rmse)
    mlflow.log_metric("mae", lr_mae)
    mlflow.log_metric("r2", lr_r2)

    mlflow.spark.log_model(lr_model, "linear_regression_model", dfs_tmpdir="/Volumes/dbacademy/default/tutorials/mlflow_tmp")

    print("Linear Regression run logged to MLflow.")

In [0]:
# MLFLOW - Step 2: Log Random Forest experiment
with mlflow.start_run(run_name="Random_Forest_EV_Charging"):
    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_param("features_used", len(feature_columns))

    mlflow.log_metric("rmse", rf_rmse)
    mlflow.log_metric("mae", rf_mae)
    mlflow.log_metric("r2", rf_r2)

    mlflow.spark.log_model(rf_model, "random_forest_model", dfs_tmpdir="/Volumes/dbacademy/default/tutorials/mlflow_tmp")

    print("Random Forest run logged to MLflow.")

## Phase 9 Summary: MLflow Experiment Tracking

**Runs logged:**
1. Linear_Regression_EV_Charging
2. Random_Forest_EV_Charging

**Comparison:**

| Metric | Linear Regression | Random Forest |
|---|---|---|
| RMSE | 21.74 | 21.88 |
| MAE | 18.68 | 18.66 |
| R² | 0.0146 | 0.0019 |

**Conclusion:**
Both models were tracked in MLflow with their parameters, metrics, and trained model artifacts, enabling
reproducible comparison. Linear Regression performed marginally better on RMSE and R², though the
difference between models is

## Phase 10: Automation

To simulate a production ML pipeline, this notebook can be wrapped into a Databricks Job — allowing it
to run automatically on a schedule (e.g., retraining as new charging session data becomes available),
without manual intervention.